# Merge the model

There are two ways to serve a LoRA model, either as:

- A merged model where you merge together the base model with the new model weights from the LoRA adapter
- Seperately so that you can change between the LoRA adapters and base model on the fly if needed

The second approach is more versatile as it allows you to easily AB test different adapters against each other and the base model, as well as serve what's essentially multiple different models very resource efficiently.  
With all that said, we are going with the merge approach here because of its simplicity.

Since we couldn't train for too long here because of running on CPU, we will use weights that have been trained a little longer (10 epochs, which would take about 1.5 hours on the resources we have in this workbench).  
You can find these weights in the folder `socratic-tutor-lora-pre-trained`.

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2-0.5B-Instruct"
LORA_DIR = "./socratic-tutor-lora-pre-trained"
OUT_DIR = "./socratic-tutor-lora-merged"

os.makedirs(OUT_DIR, exist_ok=True)

# Pick a dtype. bf16 is great if supported; otherwise use fp16.
# For CPU-only, use float32 (but it'll be slower and bigger).
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
if not torch.cuda.is_available():
    dtype = torch.float32

# Load tokenizer (always save it with the merged model)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Load base model
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    trust_remote_code=True,
)

# Load LoRA adapter on top of the base model
model = PeftModel.from_pretrained(base, LORA_DIR)

# Merge LoRA weights into the base weights and drop adapter modules
merged = model.merge_and_unload()

# (Optional) Some people like to ensure weights are contiguous before saving
merged = merged.to(dtype)

# Save merged model + tokenizer
merged.save_pretrained(OUT_DIR, safe_serialization=True)  # saves .safetensors if possible
tokenizer.save_pretrained(OUT_DIR)

print(f"✅ Merged model saved to: {OUT_DIR}")

# Saving the new model

Once you have validated that the model works as expected, let's go ahead and save it!  
We will save it as a modelcar in the OpenShift image registry and push the metadata to our model registry.

In [ ]:
from model_registry import ModelRegistry
from model_registry.utils import OCIParams

In [ ]:
username = "user1"
cluster_domain = "apps.cluster-bdcx4.bdcx4.sandbox3316.opentlc.com"
version = "0.0.3"

model_registry_url = f"https://{username}-registry-rest.{cluster_domain}"
print(f"registering to {model_registry_url}")

In [ ]:
%%bash
oc registry login \
  --to=/tmp/ocp-auth.json

In [ ]:
%env REGISTRY_AUTH_FILE=/tmp/ocp-auth.json

In [ ]:
from model_registry.utils import OCIParams
oci_upload_params = OCIParams(
    base_image="busybox",
    oci_ref=f"default-route-openshift-image-registry.{cluster_domain}/{username}-canopy/socratic-model:{version}"
)

mr = ModelRegistry(model_registry_url, author="you")

registered_model = mr.upload_artifact_and_register_model(
    name="socratic-model",
    model_files_path=OUT_DIR,
    author=username,
    version=version,
    upload_params=oci_upload_params,
    model_format_name="vllm",
    model_format_version="1"
)


Now that the model is saved and registered, we can move on to the next chapter: **Deploy To Canopy** 🚀